# Statistical Analysis: A/B-тест рассылки

Для демонстрации статистического анализа используется отдельный от RetailRocket датасет [Hillstrom E-Mail Analytics Challenge](https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html). В эксперименте клиентов случайно распределили между группами рассылки и контрольной группой без письма. Результаты отслеживали в течение двух недель.

Сравниваем группы Womens E-Mail и No E-Mail. Единица анализа - клиент; основная метрика - доля клиентов, посетивших сайт ('visit = 1'). Проверим гипотезу о равенстве этой доли в двух группах и оценим разницу между ними. Результаты этого эксперимента не относятся к пользователям RetailRocket.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../data/raw/hillstrom_email.csv')

In [3]:
df.shape

(64000, 12)

In [4]:
df.head()

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


In [5]:
df['segment'].value_counts()

segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

In [6]:
df['visit'].value_counts(dropna=False)

visit
0    54606
1     9394
Name: count, dtype: int64

In [7]:
new_df = df[df['segment'].isin(['Womens E-Mail', 'No E-Mail'])]
new_df.shape

(42693, 12)

### Число клиентов

In [8]:
new_df['segment'].value_counts()

segment
Womens E-Mail    21387
No E-Mail        21306
Name: count, dtype: int64

### Число посетивших сайт

In [9]:
new_df.groupby('segment').agg(count_visit = ('visit', 'sum'))

,count_visit
segment,
No E-Mail,2262
Womens E-Mail,3238


### Среднее

In [10]:
new_df.groupby('segment')['visit'].mean()

segment
No E-Mail        0.106167
Womens E-Mail    0.151400
Name: visit, dtype: float64

### Доли посетивших сайт

In [11]:
visit_rate = new_df.groupby('segment')['visit'].mean()
visit_rate.mul(100).round(2)

segment
No E-Mail        10.62
Womens E-Mail    15.14
Name: visit, dtype: float64

### Абсолютная разница долей

In [12]:
difference_pp = (visit_rate['Womens E-Mail'] - visit_rate['No E-Mail']) * 100
round(difference_pp, 2)

np.float64(4.52)

H₀: доля посетивших сайт одинакова в Womens E-Mail и No E-Mail.

H₁: доли различаются.

α = 0,05 - выбранный порог для проверки гипотезы.

In [13]:
new_df

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0
5,6,2) $100 - $200,134.83,0,1,Surburban,0,Phone,Womens E-Mail,1,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
63989,10,3) $200 - $350,304.30,1,1,Surburban,0,Web,Womens E-Mail,1,0,0.0
63990,6,1) $0 - $100,80.02,0,1,Surburban,0,Phone,No E-Mail,0,0,0.0
63991,1,3) $200 - $350,306.10,1,0,Surburban,1,Phone,Womens E-Mail,0,0,0.0
63993,4,4) $350 - $500,374.07,0,1,Surburban,0,Phone,Womens E-Mail,0,0,0.0


In [14]:
group_stats = new_df.groupby('segment').agg(sum_visit = ('visit', 'sum'), count_group = ('visit', 'count'))
group_stats

,sum_visit,count_group
segment,,
No E-Mail,2262,21306
Womens E-Mail,3238,21387


In [15]:
pooled_rate = group_stats["sum_visit"].sum() / group_stats["count_group"].sum()
pooled_rate

np.float64(0.12882673974656267)

In [16]:
round(pooled_rate * 100, 2)

np.float64(12.88)

In [17]:
n_womens = group_stats.loc["Womens E-Mail", "count_group"]
n_womens

np.int64(21387)

In [18]:
n_control = group_stats.loc["No E-Mail", "count_group"]
n_control

np.int64(21306)

### Стандартная ошибка разницы долей при H₀

In [19]:
se_null = (
    pooled_rate * (1 - pooled_rate)
    * (1 / n_womens + 1 / n_control)
) ** 0.5

se_null

np.float64(0.0032427070596437644)

### z-статистика

In [20]:
visit_rate

segment
No E-Mail        0.106167
Womens E-Mail    0.151400
Name: visit, dtype: float64

In [21]:
gr_letter = visit_rate.loc['Womens E-Mail']
gr_no_letter = visit_rate.loc['No E-Mail']

In [22]:
z_stat = (gr_letter - gr_no_letter) / se_null

### Двустороннее p-value

In [23]:
import math

In [24]:
math.erfc(abs(z_stat) / math.sqrt(2))

3.182403159334925e-44

In [25]:
from statsmodels.stats.proportion import proportions_ztest

groups = group_stats.loc[["Womens E-Mail", "No E-Mail"]]

z_stat_lib, p_value_lib = proportions_ztest(
    count=groups["sum_visit"],
    nobs=groups["count_group"],
    alternative="two-sided"
)

print(z_stat_lib, p_value_lib)

13.949180655258505 3.1824031593349214e-44


При уровне значимости α = 0,05 отвергаем гипотезу о равенстве долей посещений: двустороннее p-value ≈ 3,18 × 10⁻⁴⁴. В этом эксперименте доля клиентов, посетивших сайт после письма Womens E-Mail, выше, чем в контрольной группе No E-Mail, на 4,52 процентного пункта. 

### Стандартная ошибка

In [26]:
se_diff = (gr_letter * (1 - gr_letter) / n_womens + gr_no_letter * (1 - gr_no_letter) / n_control
) ** 0.5
se_diff

np.float64(0.0032343863583279126)

### Доверительный интервал

In [27]:
lower = (gr_letter - gr_no_letter - 1.96 * se_diff) * 100
upper = (gr_letter - gr_no_letter + 1.96 * se_diff) * 100
lower, upper

(np.float64(3.8893709324730277), np.float64(5.157250384937569))

Доля посещений в группе Womens E-Mail выше, чем в контрольной группе No E-Mail, на 4,52 процентного пункта. Приближённый 95% доверительный интервал для разницы составляет от 3,89 до 5,16 процентного пункта. Интервал не включает ноль; результат согласуется с отклонением гипотезы о равенстве долей по z-тесту.

### Дополнительная метрика conversion

In [28]:
new_df

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0
5,6,2) $100 - $200,134.83,0,1,Surburban,0,Phone,Womens E-Mail,1,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
63989,10,3) $200 - $350,304.30,1,1,Surburban,0,Web,Womens E-Mail,1,0,0.0
63990,6,1) $0 - $100,80.02,0,1,Surburban,0,Phone,No E-Mail,0,0,0.0
63991,1,3) $200 - $350,306.10,1,0,Surburban,1,Phone,Womens E-Mail,0,0,0.0
63993,4,4) $350 - $500,374.07,0,1,Surburban,0,Phone,Womens E-Mail,0,0,0.0


In [29]:
conversion_df = new_df.groupby('segment').agg(conversion_sum = ("conversion", 'sum'), count_clients = ('conversion', 'count'), share_conversion = ('conversion', 'mean'))
conversion_df

,conversion_sum,count_clients,share_conversion
segment,,,
No E-Mail,122,21306,0.005726
Womens E-Mail,189,21387,0.008837


In [30]:
conversion_diff = conversion_df.loc['Womens E-Mail','share_conversion'] - conversion_df.loc['No E-Mail','share_conversion']
conversion_diff * 100

np.float64(0.3111057511085258)

In [31]:
share_conv_control = conversion_df.loc['No E-Mail','share_conversion']
share_conv_letter = conversion_df.loc['Womens E-Mail','share_conversion']
share_conv_control

np.float64(0.005726086548390125)

### z-тест

In [32]:
pooled_conversion_rate = conversion_df['conversion_sum'].sum() / conversion_df['count_clients'].sum()
pooled_conversion_rate

np.float64(0.007284566556578362)

стандариная ошибка 

In [33]:
se_conversion_null = (pooled_conversion_rate * (1 - pooled_conversion_rate)* (1 / n_womens + 1 / n_control)) ** 0.5
se_conversion_null * 100

np.float64(0.08231266925873199)

### z-статистика

In [34]:
z_conversion = conversion_diff / se_conversion_null
z_conversion

np.float64(3.77956095835785)

### Двусторонее p-value

In [35]:
p_value = math.erfc(abs(z_conversion) / math.sqrt(2))
p_value

0.00015710512910668492

### Стандатрная ошибка разницы долей

In [36]:
se_conversion_diff = (share_conv_letter * (1 - share_conv_letter) / n_womens
+ share_conv_control * (1 - share_conv_control) / n_control) ** 0.5
se_conversion_diff

np.float64(0.0008226578153482761)

### Доверительный интервал

In [37]:
lower_conversion_pp = (conversion_diff - 1.96 * se_conversion_diff) * 100
upper_conversion_pp = (conversion_diff + 1.96 * se_conversion_diff) * 100
(lower_conversion_pp,upper_conversion_pp)

(np.float64(0.14986481930026369), np.float64(0.47234668291678794))

Доля клиентов с conversion = 1 составила 0,88% в группе Womens E-Mail и 0,57% в контрольной группе No E-Mail. Разница - 0,31 процентного пункта. Двусторонний z-тест дал p-value ≈ 0,000157; приближённый 95% доверительный интервал для разницы составляет от 0,15 до 0,47 процентного пункта. Таким образом, в этом эксперименте различие наблюдается и для дополнительной метрики conversion.

## Итоги и ограничения

В эксперименте Hillstrom доля посещений сайта составила 15,14% в группе Womens E-Mail против 10,62% в контрольной группе No E-Mail. Разница по основной метрике visit - 4,52 процентного пункта (приближённый 95% доверительный интервал: 3,89–5,16 п. п.; p ≈ 3,18 × 10⁻⁴⁴).

По дополнительной метрике conversion доли составили 0,88% и 0,57% соответственно. Разница - 0,31 процентного пункта (приближённый 95% доверительный интервал: 0,15-0,47 п. п.; p ≈ 0,000157).

Анализ сравнивает только две группы эксперимента. Для доверительных интервалов использовано нормальное приближение; поправка на проверку нескольких метрик не применялась. Результаты относятся к эксперименту с рассылкой и не переносятся на данные RetailRocket.